### GPUが使えるか確認

もしセルの実行結果がFalseの場合は、ランタイムを変更してください。

In [ ]:
import torch
torch.cuda.is_available()

### モデルの定義

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
  def __init__(self):
    super(Model, self).__init__()
    self.conv1_1 = nn.Conv2d( 3, 64, 3, padding=1)
    self.bn1_1 = nn.BatchNorm2d(64)
    self.conv1_2 = nn.Conv2d(64, 64, 3, padding=1)
    self.bn1_2 = nn.BatchNorm2d(64)
    self.conv1_3 = nn.Conv2d(64, 64, 3, padding=1)
    self.bn1_3 = nn.BatchNorm2d(64)
    self.conv1_4 = nn.Conv2d(64, 64, 3, padding=1, stride=2)
    self.bn1_4 = nn.BatchNorm2d(64)

    self.conv2_1 = nn.Conv2d(64, 128, 3, padding=1)
    self.bn2_1 = nn.BatchNorm2d(128)
    self.conv2_2 = nn.Conv2d(128, 128, 3, padding=1)
    self.bn2_2 = nn.BatchNorm2d(128)
    self.conv2_3 = nn.Conv2d(128, 128, 3, padding=1)
    self.bn2_3 = nn.BatchNorm2d(128)
    self.conv2_4 = nn.Conv2d(128, 128, 3, padding=1, stride=2)
    self.bn2_4 = nn.BatchNorm2d(128)

    self.conv3_1 = nn.Conv2d(128, 256, 3, padding=1)
    self.bn3_1 = nn.BatchNorm2d(256)
    self.conv3_2 = nn.Conv2d(256, 256, 3, padding=1)
    self.bn3_2 = nn.BatchNorm2d(256)
    self.conv3_3 = nn.Conv2d(256, 256, 3, padding=1)
    self.bn3_3 = nn.BatchNorm2d(256)
    self.conv3_4 = nn.Conv2d(256, 256, 3, padding=1, stride=2)
    self.bn3_4 = nn.BatchNorm2d(256)

    self.fc4 = nn.Linear(256, 256)
    self.fc5 = nn.Linear(256, 256)
    self.fc6 = nn.Linear(256, 10)

  def forward(self, x):
    x = F.relu(self.bn1_1(self.conv1_1(x)))
    x = F.relu(self.bn1_2(self.conv1_2(x)))
    x = F.relu(self.bn1_3(self.conv1_3(x)))
    x = F.relu(self.bn1_4(self.conv1_4(x)))

    x = F.relu(self.bn2_1(self.conv2_1(x)))
    x = F.relu(self.bn2_2(self.conv2_2(x)))
    x = F.relu(self.bn2_3(self.conv2_3(x)))
    x = F.relu(self.bn2_4(self.conv2_4(x)))

    x = F.relu(self.bn3_1(self.conv3_1(x)))
    x = F.relu(self.bn3_2(self.conv3_2(x)))
    x = F.relu(self.bn3_3(self.conv3_3(x)))
    x = F.relu(self.bn3_4(self.conv3_4(x)))

    x = torch.mean(x.view(x.size(0), x.size(1), -1), dim=2)
    x = F.dropout(x)
    x = F.relu(self.fc4(x))
    x = F.dropout(x)
    x = F.relu(self.fc5(x))
    return self.fc6(x)


### 学習ループの定義

In [ ]:
import os

def train(model, optimizer, criterion, trainloader, testloader, writer, val=10, model_path=None):
  counter = 0
  for epoch in range(20):

    model.train()

    running_loss = 0.0
    running_corr = 0
    running_num  = 0

    for i, data in enumerate(trainloader, 0):

      inputs, labels = data
      inputs = inputs.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()

      outputs = model(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      running_corr += (torch.max(outputs.data, 1)[1] == labels).sum().item()
      running_num  += outputs.size()[0]
      if counter % 10 == 9:
          writer.add_scalar('train/loss', running_loss/10, counter+1)
          writer.add_scalar('train/accuracy', running_corr/running_num, counter+1)
          running_loss = 0.0
          running_corr = 0
          running_num  = 0
      counter += 1

      if epoch % val == val-1:
          model.eval()
          accum_loss = 0.0
          accum_corr = 0
          accum_num  = 0
          with torch.no_grad():
              for i, data in enumerate(testloader, 0):
                  inputs, labels = data
                  inputs = inputs.to(device)
                  labels = labels.to(device)

                  outputs = model(inputs)
                  loss = criterion(outputs, labels)

                  accum_loss += loss.item()
                  accum_corr += (torch.max(outputs.data, 1)[1] == labels).sum().item()
                  accum_num  += outputs.size()[0]

              writer.add_scalar('test/loss', accum_loss/i, counter)
              writer.add_scalar('test/accuracy', accum_corr/accum_num, counter)
              print('epoch:', epoch+1, 'test loss:', accum_loss/i, 'test accuracy:', accum_corr/accum_num)

          model.train()
          if model_path is not None:
            os.makedirs(model_path, exist_ok=True)
            torch.save(model.state_dict(), model_path + '/%05d.pth' % (epoch+1))

### モデルの学習

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import datetime

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    ),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    ),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
    )

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=512,
    shuffle=True,
    num_workers=2
    )

testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
    )

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=512,
    shuffle=False,
    num_workers=2
    )

model = Model()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
  model.parameters(),
  lr=0.1,
  momentum=0.9,
  weight_decay=0.0001)

timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

model_dir = './models'
log_dir = './logs/cifar10_' + timestamp
writer = SummaryWriter(log_dir=log_dir)

train(
  model,
  optimizer,
  criterion,
  trainloader,
  testloader,
  writer,
  val=1,
  model_path=model_dir)

### 学習したモデルの確認

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

import numpy as np
import datetime
import matplotlib.pyplot as plt

%matplotlib inline

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
      (0.4914, 0.4822, 0.4465),
      (0.2023, 0.1994, 0.2010)),
])

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False,
    download=True,
    transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=1,
    shuffle=False,
    num_workers=2)

model = Model()

model_sd = torch.load('models/00019.pth')
model.load_state_dict(model_sd)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

model.eval()

dataiter = iter(testloader)

cnt = 0
for x, l in dataiter:
    x = x.to(device)
    y = F.softmax(model(x), dim=1)
    y = y.to('cpu')
    print(y.detach().numpy())
    print(l.item())

    img = x.to('cpu').numpy().reshape(
            (3, 32, 32)).transpose((1,2,0))
    img = img * np.array(
            (0.2023, 0.1994, 0.2010)).reshape((1,1,-1))
    img = img + np.array(
            (0.4914, 0.4822, 0.4465)).reshape((1,1,-1))

    plt.imshow(img)
    plt.show()
    cnt += 1
    if cnt == 4:
        break
